In [ ]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
from analysis.loaders import (load_session, load_trials, load_stims,
                              session_info, load_video_timestamps)

In [2]:
data_dir = Path("/Users/hakan/VRFarm/data/HK001/HK001_20260603/HK001_20260603_001/")

In [5]:
# Per-trial structured array (all scalar datasets)
trials = load_trials(data_dir)
trials.dtype.names

('adaptive_state',
 'block_num',
 'contrast',
 'display_latency_s',
 'first_lick_t',
 'iti_duration_s',
 'iti_lick_count',
 'iti_start_t',
 'level_effective',
 'outcome_t',
 'response_window_t',
 'reward_amount_ul',
 'reward_t',
 'stim_az_deg',
 'stim_onset_t',
 'sync_ok',
 'trial_num',
 'trial_outcome',
 'true_onset_t')

In [12]:
# Full session data (includes vlen arrays like lick_times)
data = load_session(data_dir)
[print(d) for d in data.keys()]

_iti_durations_planned
adaptive_state
block_num
contrast
display_latency_s
first_lick_t
iti_duration_s
iti_lick_count
iti_lick_times
iti_start_t
level_effective
lick_times
outcome_t
response_window_t
reward_amount_ul
reward_t
stim_az_deg
stim_onset_t
sync_ok
trial_num
trial_outcome
true_onset_t


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [ ]:
# Video frame timestamps (frame_timestamps.npy)
# t        = host wall clock per frame (same timebase as the HDF5 events)
# t_sensor = hardware SensorTimestamp mapped to wall clock (None on old 2-col files)
vts = load_video_timestamps(data_dir)
print(f"{len(vts['frame_idx'])} frames @ {vts['avg_fps']:.2f} fps avg | "
      f"sensor clock: {'yes' if vts['t_sensor'] is not None else 'no (old format)'}")

# Frame-interval sanity check: wall clock is jittery, sensor clock should be flat
import numpy as np
import matplotlib.pyplot as plt
t = vts["t_sensor"] if vts["t_sensor"] is not None else vts["t"]
plt.figure(figsize=(8, 2.5))
plt.plot(np.diff(t) * 1e3, lw=0.5)
plt.xlabel("frame"); plt.ylabel("Δt (ms)"); plt.title("frame intervals")
plt.tight_layout()

# Align a frame to behavior, e.g. frames within ±1 s of the first stim onset:
# trials = load_trials(data_dir)
# on = trials["stim_onset_t"][0]
# frames = vts["frame_idx"][(t > on - 1) & (t < on + 1)]